# `meatrider` on a Colab T4

The tier is SHA-256d over a real 80-byte Bitcoin block header, and it carries one
kernel per optimisation the repo has: portable C, SHA-NI, AVX2, AVX-512, OpenCL,
Vulkan, WebGPU, wasm SIMD, PTX — and CUDA, which is the one this notebook exists
for, because the machine that wrote it has no NVIDIA device.

**Runtime → Change runtime type → T4 GPU**, then *Runtime → Run all*. About four
minutes.

Every row is checked twice: the digest is compared against an independent Python
implementation of the header (`tools/meatrider_ref.py`), and then each kernel
mines a real rung whose nonce the C reference re-derives. A fast wrong answer
fails the run.


In [ ]:
import subprocess
print(subprocess.run("nvidia-smi -L; nvcc --version | tail -2", shell=True,
                     capture_output=True, text=True).stdout)
cpu = open("/proc/cpuinfo").read()
print("cpu:", [l.split(":", 1)[1].strip() for l in cpu.splitlines() if l.startswith("model name")][0])
flags = set()
for line in cpu.splitlines():
    if line.startswith("flags"):
        flags |= set(line.split(":", 1)[1].split())
print("isa:", sorted(f for f in flags if f.startswith(("avx", "sha_"))))


In [ ]:
%cd /content
!rm -rf anubis-solver
!git clone --depth 1 https://github.com/celikfana-lab/anubis-solver.git
%cd anubis-solver
!git log --oneline -1


In [ ]:
# sm_75 is the T4; nvcc builds every .cu in the tree, including the meatrider one.
!make -j4 CUDA_ARCH=sm_75 2>&1 | tail -12


In [ ]:
!python3 -c "import anubis_solver as a; s=a.AnubisSolver(); print('compiled: ', s.backends()); print('gpu: ', s.gpu_info())"
!python3 tests/test_meatrider.py | tail -3


In [ ]:
# Every available kernel against the Python reference, then a real mine each.
!python3 tests/test_meatrider_backends.py


In [ ]:
!python3 tools/meatrider_backends.py --threads 1 --reps 7 --json results/colab.json 2>&1 | tail -30


In [ ]:
import json
d = json.load(open("results/colab.json"))
print(f"cuda devices: {d.get('cuda_devices')}")
for r in d["rows"]:
    print(f"{r['backend']:8s} {r['hashes_per_second']/1e6:12.2f} MHash/s   "
          f"2^{r['difficulty']}, {r['solves']} solves, {r['threads']} thread(s)")


## Optional: the TPU runtime

Colab's *TPU* runtime is a different machine type; you can only have one
accelerator at a time. The repo's TPU backend serves the `sha256` tier
(`src/google/tpu/`), and `meatrider` deliberately has no TPU kernel: a PJRT
SHA-256 program measures 9.38 MHash/s, so two compressions would land near 4.7 —
slower than the AVX2 kernel on the same host (see `src/meatrider/README.md`).

This cell only checks that the runtime the repo's TPU path needs is present, so
the number above is not compared against a broken dependency.


In [ ]:
!python3 -c "import ctypes; ctypes.CDLL('libtpu.so'); print('libtpu: present')" 2>&1 | tail -2
!make test-tpu 2>&1 | tail -8
